# LogiScan — Unified Classifier Training (v2)
## Single DeBERTa-v3 with Dual Heads (Coarse + Fine)

**Uses:** `unified_training_data.json` (Includes synthetic and multi-turn data)
**Output:** `unified_classifier.zip` — replaces separate Stage 2 & Stage 3 models

In [ ]:
!pip install -q transformers torch scikit-learn tqdm

In [ ]:
import json
import os
from pathlib import Path

import torch
import torch.nn as nn
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# Upload your dataset
from google.colab import files

print("📁 Upload data/unified_training_data.json:")
uploaded = files.upload()

data_file = "unified_training_data.json"
if not os.path.exists(data_file):
    # Handle potential renaming during upload
    data_file = list(uploaded.keys())[0]

with open(data_file) as f:
    data = json.load(f)

print(f"Loaded {len(data)} samples")

# Build fine-grained labels dynamically from whatever classes are in the uploaded data
FINE_LABELS = sorted(set(d["fallacy"] for d in data))
fine2id = {l: i for i, l in enumerate(FINE_LABELS)}
id2fine = {i: l for l, i in fine2id.items()}

# Map fine → coarse
FINE_TO_COARSE = {
    "affirming_consequent": "Formal",
    "denying_antecedent": "Formal",
    "false_dilemma": "Formal",
    "ad_hominem": "Informal (Relevance)",
    "straw_man": "Informal (Relevance)",
    "appeal_to_emotion": "Informal (Relevance)",
    "appeal_to_authority": "Informal (Relevance)",
    "red_herring": "Informal (Relevance)",
    "bandwagon": "Informal (Relevance)",
    "hasty_generalization": "Informal (Relevance)",
    "moving_goalposts": "Informal (Relevance)",
    "tu_quoque_contextual": "Informal (Relevance)",
    "tu_quoque": "Informal (Relevance)",
    "begging_the_question": "Informal (Presumption)",
    "false_cause": "Informal (Presumption)",
    "slippery_slope": "Informal (Presumption)",
    "appeal_to_nature": "Informal (Presumption)",
    "appeal_to_tradition": "Informal (Presumption)",
    "no_true_scotsman": "Informal (Presumption)",
    "equivocation": "Informal (Ambiguity)",
    "composition": "Informal (Presumption)",
    "division": "Informal (Presumption)"
}

COARSE_LABELS = sorted(set(FINE_TO_COARSE.values()))
coarse2id = {l: i for i, l in enumerate(COARSE_LABELS)}
id2coarse = {i: l for l, i in coarse2id.items()}

print(f"Fine classes: {len(FINE_LABELS)} | Coarse classes: {len(COARSE_LABELS)}")

# Build tensors (only for samples with mapped labels)
texts = [d["text"] for d in data if d["fallacy"] in FINE_TO_COARSE]
fine_ids = [fine2id[d["fallacy"]] for d in data if d["fallacy"] in FINE_TO_COARSE]
coarse_ids = [coarse2id[FINE_TO_COARSE[d["fallacy"]]] for d in data if d["fallacy"] in FINE_TO_COARSE]

if len(texts) < len(data):
    missing = set(d["fallacy"] for d in data) - set(FINE_TO_COARSE.keys())
    print(f"⚠️ Dropped {len(data) - len(texts)} samples due to unmapped labels: {missing}")

In [ ]:
# Split
X_train, X_temp, yf_train, yf_temp, yc_train, yc_temp = train_test_split(
    texts, fine_ids, coarse_ids, test_size=0.2, random_state=42, stratify=fine_ids
)
X_val, X_test, yf_val, yf_test, yc_val, yc_test = train_test_split(
    X_temp, yf_temp, yc_temp, test_size=0.5, random_state=42, stratify=yf_temp
)
print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

In [ ]:
# Dataset
class DualDataset(Dataset):
    def __init__(self, texts, fine_labels, coarse_labels, tokenizer, max_len=256):
        self.texts = texts
        self.fine_labels = fine_labels
        self.coarse_labels = coarse_labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, padding="max_length", max_length=self.max_len, return_tensors="pt")
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "fine_label": torch.tensor(self.fine_labels[idx], dtype=torch.long),
            "coarse_label": torch.tensor(self.coarse_labels[idx], dtype=torch.long),
        }

tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")
train_ds = DualDataset(X_train, yf_train, yc_train, tokenizer)
val_ds = DualDataset(X_val, yf_val, yc_val, tokenizer)
test_ds = DualDataset(X_test, yf_test, yc_test, tokenizer)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16)
test_loader = DataLoader(test_ds, batch_size=16)
print(f"Batches — Train: {len(train_loader)}, Val: {len(val_loader)}")

In [ ]:
# Model: DeBERTa-v3-base with dual classification heads
class DebertaDualHead(nn.Module):
    def __init__(self, model_name, num_coarse, num_fine):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.coarse_head = nn.Linear(hidden, num_coarse)
        self.fine_head = nn.Linear(hidden, num_fine)

    def forward(self, input_ids, attention_mask, fine_label=None, coarse_label=None):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = out.last_hidden_state[:, 0, :]  # CLS token
        pooled = self.dropout(pooled)
        coarse_logits = self.coarse_head(pooled)
        fine_logits = self.fine_head(pooled)
        loss = None
        if fine_label is not None and coarse_label is not None:
            loss_fine = nn.CrossEntropyLoss()(fine_logits, fine_label)
            loss_coarse = nn.CrossEntropyLoss()(coarse_logits, coarse_label)
            loss = 0.3 * loss_coarse + 0.7 * loss_fine  # Weight fine head more
        return {"loss": loss, "coarse_logits": coarse_logits, "fine_logits": fine_logits}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DebertaDualHead("microsoft/deberta-v3-base", len(COARSE_LABELS), len(FINE_LABELS)).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Train
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
epochs = 4
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=total_steps//10, num_training_steps=total_steps)
best_f1 = 0

for epoch in range(epochs):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    for batch in pbar:
        out = model(batch["input_ids"].to(device), batch["attention_mask"].to(device), batch["fine_label"].to(device), batch["coarse_label"].to(device))
        out["loss"].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += out["loss"].item()
        pbar.set_postfix({"loss": f"{out['loss'].item():.3f}"})

    # Validate fine head
    model.eval()
    preds, truths = [], []
    with torch.no_grad():
        for batch in val_loader:
            out = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            preds.extend(torch.argmax(out["fine_logits"], dim=1).cpu().numpy())
            truths.extend(batch["fine_label"].numpy())
    f1 = f1_score(truths, preds, average="macro")
    print(f"Epoch {epoch+1}: loss={total_loss/len(train_loader):.4f}, fine_f1={f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        Path("unified_classifier").mkdir(exist_ok=True)
        torch.save(model.state_dict(), "unified_classifier/pytorch_model.bin")
        tokenizer.save_pretrained("unified_classifier")
        # Save config
        import json
        config = {
            "fine_labels": FINE_LABELS,
            "coarse_labels": COARSE_LABELS,
            "fine_to_coarse": FINE_TO_COARSE,
            "model_name": "microsoft/deberta-v3-base"
        }
        with open("unified_classifier/config.json", "w") as f: json.dump(config, f)
        print(f"  ✅ Saved (f1={f1:.4f})")

print(f"\nBest fine F1: {best_f1:.4f}")

In [ ]:
# Test set evaluation
model.eval()
fine_preds, fine_truths = [], []
coarse_preds, coarse_truths = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        out = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        fine_preds.extend(torch.argmax(out["fine_logits"], dim=1).cpu().numpy())
        fine_truths.extend(batch["fine_label"].numpy())
        coarse_preds.extend(torch.argmax(out["coarse_logits"], dim=1).cpu().numpy())
        coarse_truths.extend(batch["coarse_label"].numpy())

print("="*60)
print("FINE-GRAINED PERFORMANCE")
print("="*60)
print(classification_report(fine_truths, fine_preds, target_names=FINE_LABELS, zero_division=0))
print(f"Macro F1: {f1_score(fine_truths, fine_preds, average='macro'):.4f}")

print("\n"+"="*60)
print("COARSE PERFORMANCE")
print("="*60)
print(classification_report(coarse_truths, coarse_preds, target_names=COARSE_LABELS, zero_division=0))
print(f"Macro F1: {f1_score(coarse_truths, coarse_preds, average='macro'):.4f}")

In [ ]:
# Download
!zip -r unified_classifier.zip unified_classifier/
from google.colab import files

files.download("unified_classifier.zip")
print("\n📥 Download complete!")